# Урок 3. Подготовка данных и чанкинг

Первый компонент RAG-пайплайна: режем документацию Ollama на смысловые фрагменты.

## Функция чанкинга

Режем по заголовкам `##`, длинные секции дробим по параграфам,
слишком короткие куски (одни заголовки без содержания) выбрасываем.

In [ ]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.

    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются

    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

## Загрузка документов

`rglob("*")` обходит папку рекурсивно, включая подпапки `api/`, `capabilities/`,
`integrations/`, `tools/`. Фильтр по расширению отсеивает `.svg`, `.css`, `.png`, `.json`, `.yaml`.

In [ ]:
from pathlib import Path

DOCS_DIR = Path("docs")

documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}")

## Нарезка на чанки

Вместе с текстом сохраняем путь к исходному файлу — в следующих уроках
это позволит ассистенту указывать источник ответа.

In [ ]:
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

In [ ]:
for chunk in chunks[:3]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

## Эксперимент: как параметры влияют на результат

Прогоняем весь датасет с разными `max_chars` и смотрим на статистику.

In [ ]:
print(f"{'max_chars':>10} | {'чанков':>7} | {'средняя':>8} | {'медиана':>8} | {'макс':>6}")
print("-" * 52)

for max_chars in (200, 400, 800, 1500, 3000):
    lengths = [
        len(c)
        for doc in documents
        for c in chunk_text(doc["text"], max_chars=max_chars)
    ]
    lengths.sort()
    median = lengths[len(lengths) // 2]
    avg = sum(lengths) // len(lengths)
    print(f"{max_chars:>10} | {len(lengths):>7} | {avg:>8} | {median:>8} | {max(lengths):>6}")